# ALP diphoton interpretation

Convert expected and observed diphoton signal limits into lower bounds on the ALP decay constant $f_a$.
The baseline uses $c_1=c_2=c_3=10$, up to one additional matrix-element jet with matching, and
NNPDF31_nnlo_as_0118_nf_4_mc_hessian (325500). Input provenance is recorded in `../data/README.md`.
Numerical consistency checks run alongside the calculation.

## From signal rates to the decay constant

For $c_1=c_2=c_3=10$, the diphoton signal rate falls as $1/f_a^2$: larger $f_a$ means a weaker signal. An upper limit on the signal rate therefore gives a **lower bound on $f_a$**, excluding smaller values.

We use the predicted cross section times diphoton branching fraction **after jet matching**, at $f_{a,0}=1$ TeV. With a $K=2$ correction to the leading-order prediction, the conversion is
$$
f_{a,\mathrm{limit}}=1\,\mathrm{TeV}\sqrt{\frac{2(\sigma\times\mathrm{BR})_{\mathrm{Theory}}}{(\sigma\times\mathrm{BR})_{\mathrm{CMS}}}}.
$$
The branching fraction is already included, and the factor two is applied once in this conversion. The theory plots below show the leading-order rates.

The expected bands describe the spread of limits expected without a signal. They do not include uncertainty in the theory prediction.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import yaml
from IPython.display import display

# Locate the repository from its root or any directory beneath it.
BASE_DIR = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "interpretation" / "data").is_dir()),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError("Run from the repository root or a directory beneath it.")
DATA_DIR = BASE_DIR / "interpretation" / "data"
OUTPUT_DIR = BASE_DIR / "interpretation" / "results"
OUTPUT_DIR.mkdir(exist_ok=True)

K_FACTOR = 2.0
FA_REFERENCE_TEV = 1.0
plt.rcParams.update({"font.size": 12, "axes.linewidth": 1.2, "figure.dpi": 120})

## Expected and observed signal limits

The observed limit measures how large a diphoton signal is allowed by the data at each mass. The expected limit describes the sensitivity in the absence of a signal; its one- and two-standard-deviation bands show the range of possible outcomes from statistical fluctuations.

We check that masses are unique and ordered and that expected bands remain positive and correctly nested.

In [ ]:
def read_limits(path, observed_name, expected_name, units):
    with path.open() as stream:
        table = yaml.safe_load(stream)
    independent = table["independent_variables"]
    if len(independent) != 1 or independent[0]["header"]["units"] != "GeV":
        raise ValueError("Expected one mass variable in GeV.")
    variables = {variable["header"]["name"]: variable for variable in table["dependent_variables"]}
    observed = variables[observed_name]
    expected = variables[expected_name]
    if any(variable["header"]["units"] != units for variable in (observed, expected)):
        raise ValueError(f"Expected limits in {units}.")
    masses = np.array([row["value"] for row in independent[0]["values"]], dtype=float)
    if not (len(masses) == len(observed["values"]) == len(expected["values"])):
        raise ValueError("Mass and limit arrays have different lengths.")
    frame = pd.DataFrame({
        "mass_GeV": masses,
        "observed": [row["value"] for row in observed["values"]],
        "expected": [row["value"] for row in expected["values"]],
    })
    for sigma in (1, 2):
        offsets = np.array([
            [
                next(error["asymerror"] for error in row["errors"] if error["label"] == f"{sigma}σ")[side]
                for side in ("minus", "plus")
            ]
            for row in expected["values"]
        ], dtype=float)
        if not (np.all(offsets.min(axis=1) <= 0) and np.all(offsets.max(axis=1) >= 0)):
            raise ValueError("Error offsets do not bracket the central value.")
        frame[f"expected_minus{sigma}sigma"] = frame["expected"] + offsets.min(axis=1)
        frame[f"expected_plus{sigma}sigma"] = frame["expected"] + offsets.max(axis=1)
    if not (np.isfinite(frame.to_numpy()).all() and (frame.to_numpy() > 0).all()):
        raise ValueError("Masses, limits, and band endpoints must be finite and positive.")
    if not np.all(np.diff(masses) > 0):
        raise ValueError("Masses must be unique and strictly increasing.")
    ordered = frame[[
        "expected_minus2sigma", "expected_minus1sigma", "expected",
        "expected_plus1sigma", "expected_plus2sigma",
    ]].to_numpy()
    if not np.all(np.diff(ordered, axis=1) >= 0):
        raise ValueError("Expected bands are not nested.")
    return frame


cms_limits = read_limits(DATA_DIR / "cross_section_limits_v1.yaml", "Obs limit", "Exp limit", "pb")
print(f"Loaded {len(cms_limits)} mass hypotheses from {cms_limits.mass_GeV.min():g} to {cms_limits.mass_GeV.max():g} GeV.")
display(cms_limits.head())

## Predict the signal rate at each mass

The theory predictions are available every 5 GeV. We connect neighboring predictions with straight lines to obtain a signal rate at each mass tested by the search, keeping the measured limits at their original masses.

The plot shows how the predicted diphoton rate changes with ALP mass. The interpolation passes through every theory point and is used only within the available 10–70 GeV range.

In [ ]:
theory = pd.read_csv(DATA_DIR / "theory_baseline_1j_matched.csv")
with (DATA_DIR / "theory_baseline_1j_matched.json").open() as stream:
    theory_metadata = json.load(stream)
assert list(theory.mass_GeV) == list(range(10, 71, 5))
assert np.isfinite(theory.to_numpy()).all() and (theory.to_numpy() > 0).all()
np.testing.assert_allclose(theory.xsec_th_pb.iloc[[0, -1]], [20.1, 7.18])
assert theory_metadata["jet_matching"] == "after matching"
assert theory_metadata["pdf_id"] == 325500
assert theory_metadata["fa_reference_TeV"] == FA_REFERENCE_TEV
print(f"Baseline: 1j after matching; {theory_metadata['pdf_set']} ({theory_metadata['pdf_id']})")


def interpolate_theory(masses):
    masses = np.asarray(masses, dtype=float)
    if not np.isfinite(masses).all():
        raise ValueError("Masses must be finite.")
    if np.any((masses < theory.mass_GeV.min()) | (masses > theory.mass_GeV.max())):
        raise ValueError("Theory interpolation is restricted to 10–70 GeV.")
    return np.interp(masses, theory["mass_GeV"], theory["xsec_th_pb"])


mass = cms_limits["mass_GeV"].to_numpy()
theory_at_limits = interpolate_theory(mass)
np.testing.assert_allclose(interpolate_theory(theory.mass_GeV), theory.xsec_th_pb, rtol=0, atol=0)
np.testing.assert_allclose(interpolate_theory([12.5]), [19.55])
display(theory)

figure, axis = plt.subplots(figsize=(8, 4))
axis.plot(mass, theory_at_limits, label="Linear interpolation at CMS masses")
axis.scatter(theory.mass_GeV.to_numpy(), theory.xsec_th_pb.to_numpy(), color="black", zorder=3, label="Theory points")
axis.set(xlabel=r"$m_a$ [GeV]", ylabel=r"LO $\sigma\times\mathrm{BR}$ [pb]")
axis.legend()
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "theory_interpolation.pdf")
plt.show()

## Dependence on the decay constant

At fixed couplings, the diphoton rate scales as $1/f_a^2$. The curves use the same matched prediction
as the limit conversion. Internal checks cover the inverse-square rate law and a cubic mass-scaling fixture;
these are algebraic checks, not independent simulation validation.

In [ ]:
validation_masses = np.array([20.0, 30.0, 50.0, 70.0])
fa_scan_TeV = np.linspace(10, 30, 201)
selected_normalizations = interpolate_theory(validation_masses)
selected_cross_sections = selected_normalizations[:, None] * (FA_REFERENCE_TEV / fa_scan_TeV[None, :]) ** 2

# Internal scaling fixtures; the width normalization has arbitrary units.
validation_mass_scan = np.linspace(10, 70, 241)
validation_width = 3.13e-8 * validation_mass_scan ** 3
np.testing.assert_allclose(
    np.polyfit(np.log(validation_mass_scan), np.log(validation_width), 1)[0],
    3.0, atol=1e-12, rtol=0,
)
np.testing.assert_allclose(
    [np.polyfit(np.log(fa_scan_TeV), np.log(values), 1)[0] for values in selected_cross_sections],
    -2.0, atol=1e-12, rtol=0,
)
np.testing.assert_allclose(
    selected_cross_sections * (fa_scan_TeV[None, :] / FA_REFERENCE_TEV) ** 2,
    np.broadcast_to(selected_normalizations[:, None], selected_cross_sections.shape),
)

figure, axis = plt.subplots(figsize=(8, 4.4))
for mass_value, curve in zip(validation_masses, selected_cross_sections):
    axis.plot(fa_scan_TeV, curve, label=rf"$m_a={mass_value:g}$ GeV")
axis.set(xlabel=r"$f_a$ [TeV]", ylabel=r"LO $\sigma\times\mathrm{BR}$ [pb]")
axis.legend()
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "fa_scaling.pdf")
plt.show()

## Calculate the decay-constant limits and expected bands

We convert the observed and expected signal limits into lower bounds on $f_a$ at each mass. A smaller allowed signal rate gives a stronger bound on $f_a$.

The same conversion is applied to both edges of each expected band. Since the conversion reverses the ordering, the upper edge of a cross-section band becomes the lower edge of the corresponding $f_a$ band. This preserves the full asymmetric shape of the expected bands.

In [ ]:
def to_fa(cross_section_limit_pb, theory_pb):
    limits = np.asarray(cross_section_limit_pb, dtype=float)
    predictions = np.asarray(theory_pb, dtype=float)
    if not (np.isfinite(limits).all() and np.isfinite(predictions).all()):
        raise ValueError("Cross sections must be finite.")
    if np.any(limits <= 0) or np.any(predictions <= 0):
        raise ValueError("Cross sections must be positive.")
    return FA_REFERENCE_TEV * np.sqrt(K_FACTOR * predictions / limits)


results = pd.DataFrame({"mass_GeV": mass, "xsec_th_pb": theory_at_limits})
for column in cms_limits.columns:
    if column != "mass_GeV":
        results[f"xsec_{column}_pb"] = cms_limits[column].to_numpy()

for kind in ("observed", "expected"):
    results[f"fa_{kind}_TeV"] = to_fa(cms_limits[kind], theory_at_limits)
for sigma in (1, 2):
    results[f"fa_expected_minus{sigma}sigma_TeV"] = to_fa(
        cms_limits[f"expected_plus{sigma}sigma"], theory_at_limits
    )
    results[f"fa_expected_plus{sigma}sigma_TeV"] = to_fa(
        cms_limits[f"expected_minus{sigma}sigma"], theory_at_limits
    )
    results[f"fa_expected_err_minus{sigma}sigma_TeV"] = (
        results.fa_expected_TeV - results[f"fa_expected_minus{sigma}sigma_TeV"]
    )
    results[f"fa_expected_err_plus{sigma}sigma_TeV"] = (
        results[f"fa_expected_plus{sigma}sigma_TeV"] - results.fa_expected_TeV
    )

fa_band_columns = [
    "fa_expected_minus2sigma_TeV", "fa_expected_minus1sigma_TeV", "fa_expected_TeV",
    "fa_expected_plus1sigma_TeV", "fa_expected_plus2sigma_TeV",
]
assert np.all(np.diff(results[fa_band_columns].to_numpy(), axis=1) >= 0)
for kind in ("observed", "expected"):
    np.testing.assert_allclose(
        K_FACTOR * theory_at_limits / (results[f"fa_{kind}_TeV"] / FA_REFERENCE_TEV) ** 2,
        cms_limits[kind],
    )
for sigma in (1, 2):
    for fa_side, xsec_side in (("minus", "plus"), ("plus", "minus")):
        np.testing.assert_allclose(
            K_FACTOR * theory_at_limits / (results[f"fa_expected_{fa_side}{sigma}sigma_TeV"] / FA_REFERENCE_TEV) ** 2,
            cms_limits[f"expected_{xsec_side}{sigma}sigma"],
        )
print("Interpolation, endpoint ordering, and inverse-conversion checks passed.")
display(results[["mass_GeV", "xsec_th_pb", "fa_observed_TeV"] + fa_band_columns].head(10))

## Excluded values of the decay constant

The curves show the lower bounds on $f_a$ as a function of ALP mass: values below the observed curve are excluded. The expected curve and its bands show the search sensitivity in the absence of a signal.

In [ ]:
def plot_fa_limits(masses, frame):
    figure, axis = plt.subplots(figsize=(10, 4.8))
    band2 = axis.fill_between(
        masses, frame["expected_minus2sigma"].to_numpy(), frame["expected_plus2sigma"].to_numpy(),
        color="#ffdf80", label=r"Expected $\pm2\sigma$",
    )
    band1 = axis.fill_between(
        masses, frame["expected_minus1sigma"].to_numpy(), frame["expected_plus1sigma"].to_numpy(),
        color="#80cfff", label=r"Expected $\pm1\sigma$",
    )
    observed_line, = axis.plot(masses, frame["observed"].to_numpy(), color="black", linewidth=2, label="Observed")
    expected_line, = axis.plot(
        masses, frame["expected"].to_numpy(), color="blue", linestyle="--", linewidth=2, label="Expected",
    )
    axis.set(
        xlim=(7, 73), ylim=(0.08, 250), yscale="log",
        xlabel=r"$m_a$ [GeV]", ylabel=r"$f_a$ [TeV]",
    )
    axis.text(0.04, 0.88, "CMS", transform=axis.transAxes, fontsize=23, fontweight="bold")
    axis.text(1, 1.03, r"54.4 fb$^{-1}$ (13 TeV)", transform=axis.transAxes, ha="right", fontsize=15)
    axis.text(0.95, 0.83, r"$c_1=c_2=c_3=10$", transform=axis.transAxes, ha="right", fontsize=15)
    axis.annotate("", xy=(28, 0.17), xytext=(28, 4.5),
                  arrowprops={"arrowstyle": "-|>", "color": "#ff5bb0", "lw": 1.8, "mutation_scale": 20})
    axis.text(28.5, 0.9, "Excluded", color="#ff5bb0", fontsize=15)
    axis.legend(handles=[observed_line, expected_line, band2, band1], loc="lower right", frameon=False)
    axis.xaxis.set_minor_locator(AutoMinorLocator(5))
    axis.tick_params(which="both", direction="in", top=True, right=True)
    figure.tight_layout()
    return figure, axis


computed_fa = pd.DataFrame({"mass_GeV": mass})
for column in cms_limits.columns:
    if column != "mass_GeV":
        computed_fa[column] = results[f"fa_{column}_TeV"].to_numpy()

figure, axis = plot_fa_limits(mass, computed_fa)
figure.savefig(OUTPUT_DIR / "fa_limits.pdf")
figure.savefig(OUTPUT_DIR / "fa_limits.png", dpi=180)
plt.show()

## Save the results

Save the observed and expected $f_a$ bounds, their expected bands, and the signal rates at every tested mass.
Cross sections are in pb and decay constants are in TeV. Band endpoints and distances from the expected
central value are stored separately. Results are written to `interpretation/results/`.

In [ ]:
results.to_csv(OUTPUT_DIR / "fa_limits_baseline_1j_matched.csv", index=False, float_format="%.10g")
theory.to_csv(OUTPUT_DIR / "reference_theory_baseline_1j_matched.csv", index=False)
print(f"Saved all {len(results)} mass points and plots to {OUTPUT_DIR.resolve()}")
display(results)
with (OUTPUT_DIR / "theory_baseline_metadata.json").open("w") as stream:
    json.dump(theory_metadata, stream, indent=2)